In [2]:
import requests
import mlflow

In [6]:
DATA_PROCESSOR_URI_1 = "http://127.0.0.1:8001"
DATA_PROCESSOR_URI_2 = "http://127.0.0.1:8002"

In [9]:
training_result = requests.post(f"{DATA_PROCESSOR_URI_1}/train_model",
                               json = {"local_model_id": "dp1_1",
                                       "train_data_uri": "/"})

In [10]:
training_result

<Response [200]>

In [12]:
model_uris_1 = training_result.json()
model_uris_1

{'model_server_uri': 'http://dp1:5001',
 'model_share_id': 'models:/m-192af4d38de14a38b9da25a3d1aa3400'}

In [13]:
training_results = requests.post(f"{DATA_PROCESSOR_URI_2}/train_model",
                                 json = {"local_model_id": "dp2_1",
                                         "train_data_uri": "/"})
model_uris_2 = training_results.json()
model_uris_2

{'model_server_uri': 'http://dp2:5002',
 'model_share_id': 'models:/m-0f24aaaf5f76479f8485514981c9c054'}

In [18]:
aggregator_config = {
    #"local_model_uris": [model_uris["model_server_uri"]],
    #"local_model_uris": ["http://127.0.0.1:5001"],
    "local_model_uris": ["http://dp1:5000",
                         "http://dp2:5000"],
    "share_model_ids": [model_uris_1["model_share_id"],
                        model_uris_2["model_share_id"]]
}

In [19]:
aggregator_config

{'local_model_uris': ['http://dp1:5000', 'http://dp2:5000'],
 'share_model_ids': ['models:/m-192af4d38de14a38b9da25a3d1aa3400',
  'models:/m-0f24aaaf5f76479f8485514981c9c054']}

In [20]:
aggregation_result = requests.post("http://127.0.0.1:8101/aggregate_models/",
                                   json = aggregator_config)

In [22]:
aggregation_result

<Response [200]>

In [143]:
aggregation_result.json()

{'model_server_uri': 'http://aggregator:5101',
 'model_id': 'models:/m-450658f77608436b86eb78e123e77f39'}

In [44]:
client = mlflow.MlflowClient(tracking_uri = "http://127.0.0.1:5001")

In [52]:
model_uris["model_share_id"]

'models:/m-a8b5dd6c811b4f478ddc460f7d1ce37d'

In [68]:
client.download_artifacts(model_info.model_uuid, ".")
#client.download_artifacts(model_uris["model_share_id"], ".")

RestException: RESOURCE_DOES_NOT_EXIST: Run with id=m-a8b5dd6c811b4f478ddc460f7d1ce37d not found

In [90]:
for x,y in zip(aggregator_config["local_model_uris"], aggregator_config["share_model_ids"]):
    print(x,y)

http://localhost:5000 models:/m-6d2a611b8c6f4f77956e804b2b989956
